# Projeto Olist: Limpeza e Preparação de Dados (Etapa 01)

### Objetivo: Ingestão, exploração inicial e tratamento de valores nulos/inconsistentes dos datasets da Olist.


In [137]:
# Importação das Bibliotecas
import pandas as pd
import numpy as np

In [138]:
# Configuração do Pandas para a visualização
pd.set_option('display.max_columns', None) # Vai mostrar todas as colunas
pd.set_option('display.float_format', lambda x: '%.2f' % x) # Evita a notação científica

### Dataset Orders

In [139]:
df_orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')

In [140]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [141]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


##### Aqui já consigo ver as colunas que tem valor nulo e o tipo de cada coluna

In [142]:
df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [143]:
df_orders.duplicated().sum()

np.int64(0)

#### Convertendo as colunas do dataset Orders para datetime

In [144]:
colunas_data = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

In [145]:
for col in colunas_data:
    df_orders[col] = pd.to_datetime(df_orders[col])

print("Dados após conversão: ")
display(df_orders[colunas_data].dtypes)

Dados após conversão: 


order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Criando nova coluna baseada nas regras de negócio

In [146]:
df_orders['order_complete'] = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_approved_at'].notnull()) &
    (df_orders['order_delivered_customer_date'].notnull())
)

In [147]:
display(df_orders['order_complete'].value_counts(normalize=True) * 100)

order_complete
True    97.00
False    3.00
Name: proportion, dtype: float64

In [148]:
df_orders.to_csv('../data/processed/orders_clean.csv', index=False)
print(f"Total de linhas: {len(df_orders)}")
print(f"Pedidos completos (True): {df_orders['order_complete'].sum()}")

Total de linhas: 99441
Pedidos completos (True): 96456


### Dataset Order_Items

In [149]:
df_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

In [150]:
df_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [151]:
df_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [152]:
df_items.duplicated().sum()

np.int64(0)

In [153]:
df_items.describe()

,order_item_id,price,freight_value
count,112650.00,112650.00,112650.00
mean,1.20,120.65,19.99
std,0.71,183.63,15.81
min,1.00,0.85,0.00
25%,1.00,39.90,13.08
50%,1.00,74.99,16.26
75%,1.00,134.90,21.15
max,21.00,6735.00,409.68


#### Convertendo o tipo da coluna de object para datetime

In [154]:
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

In [155]:
print(f"Total de order_ids únicos em items: {df_items['order_id'].nunique()}")
print(f"Total de product_ids únicos em items: {df_items['product_id'].nunique()}")

Total de order_ids únicos em items: 98666
Total de product_ids únicos em items: 32951


In [156]:
df_items.to_csv('../data/processed/items_clean.csv', index=False)

### Dataset Products

In [157]:
df_products = pd.read_csv('../data/raw/olist_products_dataset.csv')

In [158]:
df_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [159]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [160]:
df_products.duplicated().sum()

np.int64(0)

#### Preechendo as colunas faltantes

In [161]:
df_products['product_category_name'] = df_products['product_category_name'].fillna('sem_categoria')
df_products['product_name_lenght'] = df_products['product_name_lenght'].fillna(0)
df_products['product_description_lenght'] = df_products['product_description_lenght'].fillna(0)
df_products['product_photos_qty'] = df_products['product_photos_qty'].fillna(0)

In [162]:
colunas_fisicas = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

In [163]:
for col in colunas_fisicas:
    mediana = df_products[col].median()
    df_products[col] = df_products[col].fillna(mediana)
    print(f" {col} nulos preenchidos com mediana: {mediana}")

 product_weight_g nulos preenchidos com mediana: 700.0
 product_length_cm nulos preenchidos com mediana: 25.0
 product_height_cm nulos preenchidos com mediana: 13.0
 product_width_cm nulos preenchidos com mediana: 20.0


In [164]:
df_products.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [165]:
df_products.to_csv('../data/processed/products_clean.csv', index=False)

### Dataset Customers

In [166]:
df_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

In [167]:
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [168]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [169]:
df_customers.duplicated().sum()

np.int64(0)

In [170]:
df_customers['customer_city'].value_counts()

customer_city
sao paulo                   15540
rio de janeiro               6882
belo horizonte               2773
brasilia                     2131
curitiba                     1521
                            ...  
olhos d'agua                    1
pacotuba                        1
sao sebastiao do paraiba        1
benedito leite                  1
campos verdes                   1
Name: count, Length: 4119, dtype: int64

#### Aqui consigo ver as top cidades com mais clientes

In [171]:
print(f"Quantidade de Clientes: {df_customers['customer_id'].count()}")

Quantidade de Clientes: 99441


In [172]:
df_customers.to_csv('../data/processed/customers_clean.csv', index=False)

### Dataset Sellers

In [173]:
df_sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

In [174]:
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [175]:
df_sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [176]:
df_sellers.duplicated().sum()

np.int64(0)

In [177]:
df_sellers['seller_city'].value_counts()

seller_city
sao paulo                                 694
curitiba                                  127
rio de janeiro                             96
belo horizonte                             68
ribeirao preto                             52
                                         ... 
ipua                                        1
muqui                                       1
timoteo                                     1
pouso alegre                                1
rio de janeiro, rio de janeiro, brasil      1
Name: count, Length: 611, dtype: int64

#### Aqui consigo ver as top cidades dos vendedores

In [178]:
df_sellers.to_csv('../data/processed/sellers_clean.csv', index=False)

### Dataset Order_Payments

In [179]:
df_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

In [180]:
df_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [181]:
df_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [182]:
df_payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.00,103886.00,103886.00
mean,1.09,2.85,154.10
std,0.71,2.69,217.49
min,1.00,0.00,0.00
25%,1.00,1.00,56.79
50%,1.00,1.00,100.00
75%,1.00,4.00,171.84
max,29.00,24.00,13664.08


In [183]:
df_payments.count()

order_id                103886
payment_sequential      103886
payment_type            103886
payment_installments    103886
payment_value           103886
dtype: int64

In [184]:
df_payments.duplicated().sum()

np.int64(0)

In [185]:
df_payments.to_csv('../data/processed/payments_clean.csv', index=False)

### Dataset Order_Reviews

In [186]:
df_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

In [187]:
df_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [188]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [189]:
df_reviews.duplicated().sum()

np.int64(0)

In [190]:
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('')

### Criação do Dataset Analytics fazendo o merge dos dados

In [191]:
# Merge encadeado — construindo o dataset analítico final
df_analytics = (
    df_orders
    .merge(df_items,     on='order_id',    how='left')
    .merge(df_products,  on='product_id',  how='left')
    .merge(df_customers, on='customer_id', how='left')
    .merge(df_sellers,   on='seller_id',   how='left')
    .merge(df_reviews[['order_id', 'review_score', 'review_comment_message']],
           on='order_id', how='left')
    .merge(df_payments[['order_id', 'payment_type', 'payment_installments', 'payment_value']],
           on='order_id', how='left')
)

print(f"Linhas: {df_analytics.shape[0]}")
print(f"Colunas: {df_analytics.shape[1]}")


Linhas: 119143
Colunas: 35


In [194]:
# Mudando a coluna order_complete para int, lendo 1 ou 0
df_analytics['order_complete'] = df_analytics['order_complete'].astype(int)

In [196]:
# Remoção da coluna 'review_comment_message', pois não será utilizada
df_analytics = df_analytics.drop(columns=['review_comment_message'])

In [197]:
df_analytics.to_csv('../data/processed/analytics.csv', index=False)

In [193]:
print(df_analytics.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_complete', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'review_score', 'review_comment_message', 'payment_type', 'payment_installments', 'payment_value']
